# SMART-MUSTAHIK: Sistem Pendukung Keputusan Penyaluran Zakat Presisi Menggunakan Analisis Spasial Berbasis QS. At-Taubah: 60

**Peneliti / Penulis**: Tim Karya Tulis Ilmiah Qur'an (KTIQ)

**Deskripsi Proyek**:
Notebook ini mengimplementasikan sistem pendukung keputusan penyaluran zakat presisi (`SMART-MUSTAHIK`). Sistem ini mengintegrasikan data sosio-ekonomi rumah tangga berstandar Badan Pusat Statistik (BPS) dengan fitur geospasial kecerahan cahaya malam hari dari satelit **VIIRS (Visible Infrared Imaging Radiometer Suite) Nighttime Lights (NTL)**. Kriteria kelayakan (*ground truth*) diformulasikan berdasarkan landasan Fiqih Kifayah berbasis **QS. At-Taubah ayat 60** untuk mengklasifikasikan rumah tangga ke dalam kategori **Mustahik (Fakir/Miskin)** dan **Non-Mustahik (Mampu)**, serta mengevaluasi efektivitasnya dalam meminimalkan *Inclusion Error* dan *Exclusion Error*.

---

In [ ]:
# ==============================================================================
# CELL 1: IMPORT LIBRARIES & SETUP LINGKUNGAN EKSPERIMEN
# ==============================================================================

# 1. Mengimpor pustaka utama untuk manipulasi dan analisis data
import os                                # Operasi sistem berkas dan direktori
import pandas as pd                      # Pengolahan struktur data tabular DataFrame
import numpy as np                       # Operasi matriks dan komputasi numerik

# 2. Mengimpor IPython display untuk tampilan tabel terformat
try:
    from IPython.display import display
except ImportError:
    display = print

# 3. Mengimpor pustaka Scikit-Learn untuk Machine Learning dan evaluasi performa
from sklearn.ensemble import RandomForestClassifier         # Algoritma klasifikasi Random Forest
from sklearn.model_selection import train_test_split        # Pembagian dataset Train-Test
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, # Metrik evaluasi standar
    confusion_matrix, classification_report                 # Matriks konfusi dan laporan rinci
)

# 4. Mengimpor pustaka visualisasi grafik data
import matplotlib.pyplot as plt          # Pembuatan grafik dan tata letak plot
import seaborn as sns                    # Visualisasi data statistik berbasis Matplotlib

# 5. Mengatur random seed secara global untuk menjamin reproduksibilitas hasil eksperimen
SEED = 42
np.random.seed(SEED)

# 6. Konfigurasi direktori kerja
os.makedirs('data', exist_ok=True)
os.makedirs('images', exist_ok=True)

# 7. Konfigurasi gaya visualisasi Seaborn
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("[CELL 1 SELESAI] Semua pustaka (libraries) dan konfigurasi seed 42 berhasil dimuat.")


In [ ]:
# ==============================================================================
# CELL 2: LOAD DATASET BPS / GENERASI DATASET STRUKTUR BPS (10.000 SAMPEL)
# ==============================================================================

def muat_atau_generasi_dataset(file_path=os.path.join('data', 'data_bps.csv'), n_samples=10000):
    """
    Fungsi untuk memuat dataset BPS dari file CSV (jika ada)
    atau secara otomatis menggenerasi dataset sintetis terstruktur BPS sebanyak n_samples.
    """
    if os.path.exists(file_path):
        print(f"Memuat dataset BPS eksternal dari file '{file_path}'...")
        df = pd.read_csv(file_path)
    else:
        target_sintetis = os.path.join('data', 'data_bps_sintetis.csv')
        if os.path.exists(target_sintetis):
            print(f"Memuat dataset sintetis terstruktur BPS dari '{target_sintetis}'...")
            df = pd.read_csv(target_sintetis)
        else:
            print(f"File '{file_path}' tidak ditemukan. Menggenerasi {n_samples:,} dataset sintetis terstruktur BPS...")
            
            # 1. pendapatan_per_kapita (Rasio, Rp/bulan): Distribusi Log-Normal
            pendapatan_total = np.random.lognormal(mean=14.5, sigma=0.6, size=n_samples)
            
            # 2. jumlah_tanggungan (Jumlah orang): Distribusi Poisson rata-rata 3 orang + 1 pendamping
            tanggungan = np.random.poisson(lam=3, size=n_samples) + 1
            
            # Pendapatan per kapita = Total pendapatan / Jumlah tanggungan keluarga
            pendapatan_per_kapita = pendapatan_total / tanggungan
            
            # 3. skor_aset (Interval 0 - 100): Kepemilikan aset rumah tangga
            skor_aset = np.random.uniform(10.0, 95.0, size=n_samples)
            
            # 4. kualitas_hunian (Ordinal 1-5): Indikator fisik rumah (Atap, Dinding, Lantai)
            kualitas_hunian = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.15, 0.25, 0.30, 0.20, 0.10])
            
            # 5. akses_sanitasi (Biner 0/1): Air bersih & jamban layak
            akses_sanitasi = np.random.choice([0, 1], size=n_samples, p=[0.35, 0.65])
            
            # 6. radiasi_satelit_ntl (Kontinu, nW/cm2/sr): Intensitas cahaya malam hari satelit VIIRS
            radiasi_satelit_ntl = np.random.uniform(0.1, 50.0, size=n_samples)
            
            # Membentuk DataFrame pandas
            df = pd.DataFrame({
                'pendapatan_per_kapita': np.round(pendapatan_per_kapita, 2),
                'skor_aset': np.round(skor_aset, 2),
                'jumlah_tanggungan': tanggungan,
                'kualitas_hunian': kualitas_hunian,
                'akses_sanitasi': akses_sanitasi,
                'radiasi_satelit_ntl': np.round(radiasi_satelit_ntl, 2)
            })
            
            # Menyimpan dataset sintetis ke CSV
            df.to_csv(target_sintetis, index=False)
            print(f"Dataset sintetis berhasil dibuat dan disimpan ke '{target_sintetis}'.")
            
    return df

# Memuat atau menggenerasi dataset 10.000 sampel
df_bps = muat_atau_generasi_dataset(n_samples=10000)

# Menampilkan 5 sampel awal dataset
print("\n--- 5 Sampel Awal Dataset BPS & Geospasial Satelit ---")
display(df_bps.head())
print(f"Dimensi Dataset: {df_bps.shape[0]:,} Baris x {df_bps.shape[1]} Kolom")
print("[CELL 2 SELESAI] Dataset sosio-ekonomi BPS & Satelit NTL berhasil disiapkan.")


In [ ]:
# ==============================================================================
# CELL 3: FORMULASI LABEL MUSTAHIK BERBASIS QS. AT-TAUBAH: 60 & FIQIH KIFAYAH
# ==============================================================================

# Ambang Batas Had Kifayah (Misal: Rp 1.800.000,- per kapita per bulan)
HAD_KIFAYAH = 1800000

def formulasi_status_mustahik(row):
    """
    Formulasi Ground Truth Fiqih Kifayah berbasis QS. At-Taubah: 60:
    - Kategori Fakir : Pendapatan < 50% Had Kifayah DAN Kualitas Hunian <= 2
    - Kategori Miskin: Pendapatan < 100% Had Kifayah DAN Skor Aset < 40
    
    Output:
    - 1 : Mustahik (Berhak Menerima Zakat - Kategori Fakir atau Miskin)
    - 0 : Non-Mustahik (Mampu / Berada di atas batas kebutuhan dasar Kifayah)
    """
    pk = row['pendapatan_per_kapita']
    kh = row['kualitas_hunian']
    sa = row['skor_aset']
    
    # Evaluasi Kondisi Fakir (< 50% Had Kifayah & Hunian Buruk)
    is_fakir = (pk < 0.5 * HAD_KIFAYAH) and (kh <= 2)
    
    # Evaluasi Kondisi Miskin (< 100% Had Kifayah & Skor Aset < 40)
    is_miskin = (pk < HAD_KIFAYAH) and (sa < 40)
    
    # Pengelompokan Label Biner (1: Mustahik, 0: Non-Mustahik)
    if is_fakir or is_miskin:
        return 1
    else:
        return 0

# Menerapkan fungsi formulasi label pada dataset
df_bps['status_mustahik'] = df_bps.apply(formulasi_status_mustahik, axis=1)

# Ringkasan Hasil Formulasi Label Mustahik
print("--- DISTRIBUSI LABEL GROUND TRUTH MUSTAHIK (QS. AT-TAUBAH: 60) ---")
distrib = df_bps['status_mustahik'].value_counts()
distrib_pct = df_bps['status_mustahik'].value_counts(normalize=True) * 100

for label_code, count in distrib.items():
    nama_label = "Mustahik (Fakir/Miskin)" if label_code == 1 else "Non-Mustahik (Mampu)"
    print(f"Label {label_code} ({nama_label:23s}): {count:,} Rumah Tangga ({distrib_pct[label_code]:.2f}%)")

# Visualisasi & Simpan Grafik Distribusi
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_bps, x='status_mustahik', hue='status_mustahik', palette=['#3498db', '#e74c3c'], legend=False)
plt.title('Distribusi Status Mustahik (QS. At-Taubah: 60)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Status (0: Non-Mustahik, 1: Mustahik)', fontweight='bold')
plt.ylabel('Jumlah Rumah Tangga', fontweight='bold')
plt.xticks([0, 1], ['Non-Mustahik (Mampu)', 'Mustahik (Fakir/Miskin)'])
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{h:,} ({h/len(df_bps)*100:.1f}%)', (p.get_x() + p.get_width() / 2., h / 2),
                    ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('images', 'distribusi_mustahik.png'), dpi=300)
plt.close()

print("\n[CELL 3 SELESAI] Label target 'status_mustahik' berhasil diformulasikan.")


In [ ]:
# ==============================================================================
# CELL 4: PEMBAGIAN DATASET & PELATIHAN MODEL
# ==============================================================================

# 1. Memisahkan Fitur Prediktor (X) dan Label Target (y)
X = df_bps[[
    'pendapatan_per_kapita', 
    'skor_aset', 
    'jumlah_tanggungan', 
    'kualitas_hunian', 
    'akses_sanitasi', 
    'radiasi_satelit_ntl'
]]
y = df_bps['status_mustahik']

# 2. Membagi dataset menjadi 80% Training Set dan 20% Test Set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=SEED, 
    stratify=y
)

print(f"Ukuran Dataset Training (80%): {X_train.shape[0]:,} sampel")
print(f"Ukuran Dataset Testing  (20%): {X_test.shape[0]:,} sampel\n")

# 3. Pelatihan Model Usulan: SMART-MUSTAHIK (Random Forest Classifier)
print("Melatih Model Usulan SMART-MUSTAHIK (Random Forest Classifier)... ")
model_smart = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=SEED
)
model_smart.fit(X_train, y_train)
y_pred_smart = model_smart.predict(X_test)
print("-> Model SMART-MUSTAHIK berhasil dilatih.")

# 4. Pelatihan Simulasi Metode Konvensional (Survei Manual dengan 20% Noise/Bias)
print("Simulasi Metode Konvensional (Rule-based Survei Manual dengan Bias 20%)...")
y_pred_konv = y_test.copy().values
n_noise = int(0.20 * len(y_pred_konv))
noise_indices = np.random.choice(len(y_pred_konv), size=n_noise, replace=False)
y_pred_konv[noise_indices] = 1 - y_pred_konv[noise_indices]
print("-> Simulasi Metode Konvensional berhasil dibuat.")

print("\n[CELL 4 SELESAI] Model SMART-MUSTAHIK dan Simulasi Konvensional telah siap dievaluasi.")


In [ ]:
# ==============================================================================
# CELL 5: EVALUASI PERFORMA & PERHITUNGAN INCLUSION/EXCLUSION ERROR
# ==============================================================================

def hitung_metrik_efektivitas_zakat(y_true, y_pred, nama_metode):
    """
    Menghitung metrik klasifikasi standar serta metrik spesifik efektivitas penyaluran zakat:
    - Inclusion Error Rate (% orang mampu yang salah dapat zakat / False Positive Rate)
    - Exclusion Error Rate (% warga miskin yang terlewat tidak dapat zakat / False Negative Rate)
    """
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred) * 100
    rec = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    inclusion_error = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    exclusion_error = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0.0
    
    return {
        'Metode': nama_metode,
        'Akurasi (%)': round(acc, 2),
        'Precision (%)': round(prec, 2),
        'Recall (%)': round(rec, 2),
        'F1-Score (%)': round(f1, 2),
        'Inclusion Error (%)': round(inclusion_error, 2),
        'Exclusion Error (%)': round(exclusion_error, 2)
    }

hasil_konv = hitung_metrik_efektivitas_zakat(y_test, y_pred_konv, "Metode Konvensional (Survei Manual)")
hasil_smart = hitung_metrik_efektivitas_zakat(y_test, y_pred_smart, "SMART-MUSTAHIK (Model Usulan)")

df_evaluasi = pd.DataFrame([hasil_konv, hasil_smart]).set_index('Metode')
df_evaluasi.to_csv(os.path.join('data', 'evaluasi_performa.csv'))

print("=========================================================================")
print("   TABEL PERBANDINGAN PERFORMA METODE KONVENSIONAL VS SMART-MUSTAHIK")
print("=========================================================================")
display(df_evaluasi)

print("\n--- LAPORAN KLASIFIKASI RINCI SMART-MUSTAHIK ---")
print(classification_report(y_test, y_pred_smart, target_names=['Non-Mustahik (Mampu)', 'Mustahik (Miskin)']))
print("[CELL 5 SELESAI] Evaluasi perbandingan metrik dan targeting error berhasil diselesaikan.")


In [ ]:
# ==============================================================================
# CELL 6: VISUALISASI GRAFIK HASIL EKSPERIMEN (300 DPI KE IMAGES/)
# ==============================================================================

# 1. Plot Utama Perbandingan Performa & Error Zakat (grafik_hasil_smart_mustahik.png)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ['#e74c3c', '#2ecc71']

# Subplot 1: Akurasi & F1-Score
df_plot1 = df_evaluasi[['Akurasi (%)', 'F1-Score (%)']].reset_index().melt(
    id_vars='Metode', var_name='Metrik Performa', value_name='Persentase (%)'
)
sns.barplot(data=df_plot1, x='Metrik Performa', y='Persentase (%)', hue='Metode', ax=axes[0], palette=colors)
axes[0].set_title('Perbandingan Akurasi & F1-Score Klasifikasi', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylim(0, 115)
axes[0].set_xlabel('')
axes[0].set_ylabel('Persentase (%)', fontsize=11, fontweight='bold')
axes[0].legend(title='', loc='upper left')
for p in axes[0].patches:
    val = p.get_height()
    if val > 0:
        axes[0].annotate(f'{val:.2f}%', (p.get_x() + p.get_width() / 2., val + 1.5), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Subplot 2: Inclusion & Exclusion Error
df_plot2 = df_evaluasi[['Inclusion Error (%)', 'Exclusion Error (%)']].reset_index().melt(
    id_vars='Metode', var_name='Tipe Error Penyaluran Zakat', value_name='Persentase Error (%)'
)
sns.barplot(data=df_plot2, x='Tipe Error Penyaluran Zakat', y='Persentase Error (%)', hue='Metode', ax=axes[1], palette=colors)
axes[1].set_title('Perbandingan Inclusion Error & Exclusion Error (%)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylim(0, 30)
axes[1].set_xlabel('')
axes[1].set_ylabel('Persentase Error (%)', fontsize=11, fontweight='bold')
axes[1].legend(title='', loc='upper right')
for p in axes[1].patches:
    val = p.get_height()
    if val > 0:
        axes[1].annotate(f'{val:.2f}%', (p.get_x() + p.get_width() / 2., val + 0.5), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
nama_file_grafik = os.path.join('images', 'grafik_hasil_smart_mustahik.png')
plt.savefig(nama_file_grafik, dpi=300, bbox_inches='tight')
plt.close()

# 2. Plot Feature Importance
importances = model_smart.feature_importances_
feature_names = ['Pendapatan per Kapita', 'Skor Aset', 'Jumlah Tanggungan', 'Kualitas Hunian', 'Akses Sanitasi', 'Radiasi Satelit NTL']
df_imp = pd.DataFrame({'Fitur': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=df_imp, x='Importance', y='Fitur', hue='Fitur', palette='viridis', legend=False)
plt.title('Tingkat Kepentingan Fitur (Feature Importance) Random Forest', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Nilai Importance', fontweight='bold')
plt.ylabel('Fitur Prediktor', fontweight='bold')
for p in plt.gca().patches:
    w = p.get_width()
    plt.gca().annotate(f'{w:.4f}', (w + 0.005, p.get_y() + p.get_height()/2.), ha='left', va='center', fontsize=10, fontweight='bold')
plt.xlim(0, max(importances) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join('images', 'feature_importance.png'), dpi=300)
plt.close()

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred_smart)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-Mustahik', 'Mustahik'],
            yticklabels=['Non-Mustahik', 'Mustahik'])
plt.title('Matriks Konfusi SMART-MUSTAHIK', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Prediksi Model', fontweight='bold')
plt.ylabel('Ground Truth (Fiqih Kifayah)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('images', 'confusion_matrix.png'), dpi=300)
plt.close()

print(f"[CELL 6 SELESAI] Semua grafik 300 DPI berhasil disimpan ke direktori 'images/'.")
